In [ ]:
!pip install --upgrade gdown
import gdown

folder_url = 'https://drive.google.com/drive/folders/1s5W0Nfz9NEN8gP14tge8GuouUDXs2Ssq'
gdown.download_folder(folder_url, quiet=False, use_cookies=False)

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


Retrieving folder contents


Processing file 1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF full-shape.zip
Processing file 1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p partial.zip
Processing file 1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM rotate.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF
From (redirected): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF&confirm=t&uuid=2b676301-a854-4d4c-9cf8-07708ad33bcc
To: /content/AffordanceNet/full-shape.zip
100%|██████████| 558M/558M [00:05<00:00, 97.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p
From (redirected): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p&confirm=t&uuid=71e57c3d-721a-45a7-9b9e-ac0983ce28a7
To: /content/AffordanceNet/partial.zip
100%|██████████| 2.19G/2.19G [00:28<00:00, 77.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM
To: /content/AffordanceNet/rotate.zip
100%|██████████| 1.15M/1.15M [00:00<00:00, 138MB/s]
Download completed


['/content/AffordanceNet/full-shape.zip',
 '/content/AffordanceNet/partial.zip',
 '/content/AffordanceNet/rotate.zip']

In [ ]:
# Unzip all files in the AffordanceNet folder
!unzip -o /content/AffordanceNet/full-shape.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/partial.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/rotate.zip -d /content/AffordanceNet/

# List the directory to see the extracted content
!ls -R /content/AffordanceNet/

Archive:  /content/AffordanceNet/full-shape.zip
  inflating: /content/AffordanceNet/full_shape_train_data.pkl  
  inflating: /content/AffordanceNet/full_shape_val_data.pkl  
Archive:  /content/AffordanceNet/partial.zip
  inflating: /content/AffordanceNet/partial_train_data.pkl  
  inflating: /content/AffordanceNet/partial_val_data.pkl  
Archive:  /content/AffordanceNet/rotate.zip
  inflating: /content/AffordanceNet/rotate_val_data.pkl  
/content/AffordanceNet/:
full_shape_train_data.pkl  partial_train_data.pkl  rotate_val_data.pkl
full_shape_val_data.pkl    partial_val_data.pkl    rotate.zip
full-shape.zip		   partial.zip


In [ ]:
# Remove all zip files in the AffordanceNet directory
!rm /content/AffordanceNet/*.zip

# List the directory to confirm they are gone
!ls /content/AffordanceNet/

full_shape_train_data.pkl  partial_train_data.pkl  rotate_val_data.pkl
full_shape_val_data.pkl    partial_val_data.pkl


In [ ]:
!pip install open3d torch torchvision tqdm numpy

it not requird

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ------------------------------------------------------------
# 1. Copy of DGCNN model definition (must be identical to training)
# ------------------------------------------------------------
def knn(x, k=20):
    B, _, N = x.shape
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]
    return idx

def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None:
        idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx = idx + idx_base
    idx = idx.view(-1)
    _, num_dims, _ = x.size()
    x = x.transpose(2, 1).contiguous()
    feature = x.view(B * N, -1)[idx, :]
    feature = feature.view(B, N, k, C)
    x = x.view(B, N, 1, C).repeat(1, 1, k, 1)
    feature = torch.cat((feature - x, x), dim=3)
    feature = feature.permute(0, 3, 1, 2).contiguous()
    return feature

class DGCNNPartSeg(nn.Module):
    def __init__(self, num_classes=5, k=20):
        super().__init__()
        self.k = k
        self.conv1 = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64*2, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64*2, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Conv1d(64 + 64 + 128 + 256, 256, 1, bias=False),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, num_classes, 1)
        )

    def forward(self, x):
        B, C, N = x.shape
        edge_feat = get_graph_feature(x, k=self.k)
        x1 = self.conv1(edge_feat).max(dim=-1, keepdim=False)[0]
        edge_feat = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge_feat).max(dim=-1, keepdim=False)[0]
        edge_feat = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge_feat).max(dim=-1, keepdim=False)[0]
        x_cat = torch.cat([x1, x2, x3], dim=1)
        x_global = x_cat.max(dim=-1, keepdim=True)[0].repeat(1, 1, N)
        x_concat = torch.cat([x1, x2, x3, x_global], dim=1)
        out = self.fc(x_concat).permute(0, 2, 1)
        return out

# ------------------------------------------------------------
# 2. Copy of AffordanceMultiLabelDataset (must be identical)
# ------------------------------------------------------------
class AffordanceMultiLabelDataset(Dataset):
    def __init__(self, data_list, selected_indices, num_points=2048,
                 occlusion=True, crop_angle=120, dropout=0.3):
        self.data = data_list
        self.selected_indices = selected_indices
        self.num_points = num_points
        self.occlusion = occlusion
        self.crop_angle = crop_angle
        self.dropout = dropout

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        coords = sample['full_shape']['coordinate']
        label_dict = sample['full_shape']['label']

        masks = []
        for idx_c in self.selected_indices:
            aff = AFFORDANCE_NAMES[idx_c]
            mask = label_dict[aff].flatten()
            mask = (mask > 0.5).astype(np.float32)  # binarized
            masks.append(mask)
        labels = np.stack(masks, axis=1)

        if self.occlusion:
            coords, labels = self.simulate_occlusion(coords, labels)

        if len(coords) == 0:
            coords = sample['full_shape']['coordinate']
            masks = []
            for idx_c in self.selected_indices:
                aff = AFFORDANCE_NAMES[idx_c]
                mask = label_dict[aff].flatten()
                mask = (mask > 0.5).astype(np.float32)
                masks.append(mask)
            labels = np.stack(masks, axis=1)

        if len(coords) > self.num_points:
            pos_mask = (labels.sum(axis=1) > 0.5)
            pos_indices = np.where(pos_mask)[0]
            neg_indices = np.where(~pos_mask)[0]
            n_pos = len(pos_indices)
            n_neg_needed = max(0, self.num_points - n_pos)

            if n_neg_needed > 0 and len(neg_indices) > 0:
                neg_selected = np.random.choice(neg_indices, n_neg_needed, replace=False)
                selected_indices = np.concatenate([pos_indices, neg_selected])
            elif n_pos > 0:
                selected_indices = pos_indices
            else:
                selected_indices = np.random.choice(len(coords), self.num_points, replace=False)

            if len(selected_indices) < self.num_points:
                remaining = self.num_points - len(selected_indices)
                extra = np.random.choice(len(coords), remaining, replace=True)
                selected_indices = np.concatenate([selected_indices, extra])
        else:
            selected_indices = np.random.choice(len(coords), self.num_points, replace=True)

        coords = coords[selected_indices]
        labels = labels[selected_indices]
        return torch.tensor(coords, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32)

    def simulate_occlusion(self, points, labels):
        if len(points) == 0:
            return points, labels
        pos_mask = (labels.sum(axis=1) > 0.5)
        pos_indices = np.where(pos_mask)[0]
        neg_indices = np.where(~pos_mask)[0]

        if len(neg_indices) > 0:
            neg_points = points[neg_indices]
            centroid = np.mean(neg_points, axis=0)
            centered = neg_points - centroid
            theta = np.random.uniform(0, 2*np.pi)
            phi = np.random.uniform(0, np.pi)
            view_dir = np.array([np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)])
            norms = np.linalg.norm(centered, axis=1)
            norms[norms == 0] = 1e-8
            dirs = centered / norms[:, np.newaxis]
            angles = np.arccos(np.clip(np.dot(dirs, view_dir), -1, 1))
            mask = angles < np.radians(self.crop_angle/2)
            if self.dropout > 0:
                keep_neg = np.where(mask)[0]
                n_keep = int(len(keep_neg) * (1 - self.dropout))
                if n_keep > 0 and len(keep_neg) > 0:
                    keep_neg = np.random.choice(keep_neg, n_keep, replace=False)
                    mask = np.zeros(len(neg_points), dtype=bool)
                    mask[keep_neg] = True
            neg_points = neg_points[mask]
            neg_labels = labels[neg_indices][mask]
        else:
            neg_points = np.empty((0, 3))
            neg_labels = np.empty((0, labels.shape[1]))

        pos_points = points[pos_indices]
        pos_labels = labels[pos_indices]

        if len(pos_points) > 0:
            coords = np.concatenate([pos_points, neg_points], axis=0)
            labels = np.concatenate([pos_labels, neg_labels], axis=0)
        else:
            coords = neg_points
            labels = neg_labels
        return coords, labels

# ------------------------------------------------------------
# 3. Constants (must match training)
# ------------------------------------------------------------
AFFORDANCE_NAMES = ['grasp', 'contain', 'lift', 'openable', 'layable', 'sittable', 'support',
                    'wrap_grasp', 'pourable', 'move', 'displaY', 'pushable', 'pull', 'listen',
                    'wear', 'press', 'cut', 'stab']
SELECTED_CLASSES = ['move', 'support', 'sittable', 'contain', 'displaY']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_CLASSES = len(SELECTED_CLASSES)

# ------------------------------------------------------------
# 4. Load your test data (adjust paths as needed)
# ------------------------------------------------------------
import pickle
with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
with open("/content/AffordanceNet/full_shape_val_data.pkl", "rb") as f:
    val_data = pickle.load(f)

all_categories = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = ['Scissors', 'Hammer', 'Laptop', 'Camera']
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_categories]
test_split = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES]
print(f"Test samples (zero-shot): {len(test_split)}")

# ------------------------------------------------------------
# 5. Evaluation
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DGCNNPartSeg(num_classes=NUM_CLASSES, k=20).to(device)
model.load_state_dict(torch.load("/content/model/best_model_weighted.pth", map_location=device))
model.eval()

test_dataset_no_occ = AffordanceMultiLabelDataset(
    test_split, SELECTED_INDICES, num_points=2048,
    occlusion=False, crop_angle=0, dropout=0.0
)
test_loader_no_occ = DataLoader(test_dataset_no_occ, batch_size=8, shuffle=False, num_workers=2)

iou_per_class = torch.zeros(NUM_CLASSES).to(device)
count_per_class = torch.zeros(NUM_CLASSES).to(device)

with torch.no_grad():
    for points, labels in tqdm(test_loader_no_occ, desc="Zero-Shot (no occ)"):
        points = points.permute(0, 2, 1).to(device)
        labels = labels.to(device)
        pred = model(points)
        pred_binary = (torch.sigmoid(pred) > 0.5).long()
        labels_binary = labels.long()
        for b in range(points.shape[0]):
            for c in range(NUM_CLASSES):
                if labels_binary[b, :, c].sum() > 0:
                    inter = (pred_binary[b, :, c] & labels_binary[b, :, c]).sum().float()
                    union = (pred_binary[b, :, c] | labels_binary[b, :, c]).sum().float()
                    iou = inter / (union + 1e-8)
                    iou_per_class[c] += iou
                    count_per_class[c] += 1

avg_iou_per_class = iou_per_class / (count_per_class + 1e-8)
mean_iou = avg_iou_per_class.mean().item()

print(f"\nZero‑Shot (no occlusion) mIoU: {mean_iou:.4f}")
print("Per-class IoU:")
for idx, name in enumerate(SELECTED_CLASSES):
    print(f"  {name}: {avg_iou_per_class[idx].item():.4f}")

Test samples (zero-shot): 344


Zero-Shot (no occ): 100%|██████████| 43/43 [02:25<00:00,  3.38s/it]


Zero‑Shot (no occlusion) mIoU: 0.0001
Per-class IoU:
  move: 0.0000
  support: 0.0000
  sittable: 0.0000
  contain: 0.0000
  displaY: 0.0007


In [ ]:
import numpy as np
import open3d as o3d
import pickle

def generate_synthetic_sample():
    # Choose object type
    obj_type = np.random.choice(['mug', 'bowl', 'knife', 'laptop', 'chair'])
    part_meshes = []
    part_labels = []

    if obj_type == 'mug':
        # Body (label 0)
        r = np.random.uniform(0.15, 0.3)
        h = np.random.uniform(0.3, 0.6)
        body = o3d.geometry.TriangleMesh.create_cylinder(radius=r, height=h)
        part_meshes.append(body)
        part_labels.append(0)
        # Handle (label 1)
        # Create a curved tube (torus) and attach it
        torus_r = r * 0.8
        tube_r = np.random.uniform(0.02, 0.05)
        handle = o3d.geometry.TriangleMesh.create_torus(torus_radius=torus_r, tube_radius=tube_r)
        # Rotate so it lies in XZ plane and attach to side
        R = o3d.geometry.get_rotation_matrix_from_xyz((np.pi/2, 0, 0))
        handle.rotate(R)
        handle.translate([r + 0.02, 0, 0])
        part_meshes.append(handle)
        part_labels.append(1)

    elif obj_type == 'bowl':
        # Concave body (label 4)
        r = np.random.uniform(0.25, 0.45)
        bowl = o3d.geometry.TriangleMesh.create_sphere(radius=r)
        # Crop to keep bottom half
        bbox = o3d.geometry.AxisAlignedBoundingBox(min_bound=(-r, -r, -r), max_bound=(r, r, 0.0))
        bowl = bowl.crop(bbox)
        part_meshes.append(bowl)
        part_labels.append(4)  # concave
        # Optional rim (label 5)
        if np.random.rand() > 0.5:
            rim_r = r * 0.95
            rim = o3d.geometry.TriangleMesh.create_torus(torus_radius=rim_r, tube_radius=0.02)
            rim.translate([0, 0, 0.02])
            part_meshes.append(rim)
            part_labels.append(5)

    elif obj_type == 'knife':
        # Handle (label 1) – cylinder
        h_len = np.random.uniform(0.3, 0.6)
        h_r = np.random.uniform(0.05, 0.1)
        handle = o3d.geometry.TriangleMesh.create_cylinder(radius=h_r, height=h_len)
        handle.rotate(o3d.geometry.get_rotation_matrix_from_xyz((np.pi/2, 0, 0)))
        handle.translate([-0.15, 0, 0])
        part_meshes.append(handle)
        part_labels.append(1)
        # Blade (label 2) – wedge shape (simplified as a thin box)
        blade = o3d.geometry.TriangleMesh.create_box(width=0.02, height=0.15, depth=0.35)
        blade.translate([0.15, 0, 0])
        part_meshes.append(blade)
        part_labels.append(2)

    elif obj_type == 'laptop':
        # Base (label 0)
        base = o3d.geometry.TriangleMesh.create_box(width=0.4, height=0.02, depth=0.3)
        base.translate([0, -0.05, 0])
        part_meshes.append(base)
        part_labels.append(0)
        # Screen (label 6) – thin vertical box
        screen = o3d.geometry.TriangleMesh.create_box(width=0.38, height=0.25, depth=0.015)
        screen.rotate(o3d.geometry.get_rotation_matrix_from_xyz((np.pi/2, 0, 0)))
        screen.translate([0, 0.1, 0])
        part_meshes.append(screen)
        part_labels.append(6)

    elif obj_type == 'chair':
        # Seat (label 3 – flat surface)
        seat = o3d.geometry.TriangleMesh.create_box(width=0.4, height=0.04, depth=0.4)
        seat.translate([0, 0.2, 0])
        part_meshes.append(seat)
        part_labels.append(3)
        # Legs (4 cylinders, label 0 – background, but could be separate)
        for dx, dz in [(-0.15, -0.15), (-0.15, 0.15), (0.15, -0.15), (0.15, 0.15)]:
            leg = o3d.geometry.TriangleMesh.create_cylinder(radius=0.03, height=0.3)
            leg.translate([dx, 0.05, dz])
            part_meshes.append(leg)
            part_labels.append(0)  # legs are background/structural
        # Backrest (label 3 or 6? Let's label as flat surface for support/sittable, but could be display. Use 3 for now)
        back = o3d.geometry.TriangleMesh.create_box(width=0.38, height=0.25, depth=0.03)
        back.translate([0, 0.35, -0.18])
        part_meshes.append(back)
        part_labels.append(3)  # could also be label 6 if you want to map to display

    # Sample points from each part
    all_pts = []
    all_labels = []
    for mesh, label in zip(part_meshes, part_labels):
        # Number of points proportional to area (simplified)
        n = int(500 + np.random.randint(200))
        pcd = mesh.sample_points_uniformly(number_of_points=n)
        pts = np.asarray(pcd.points)
        all_pts.append(pts)
        all_labels.append(np.full(len(pts), label))
    pts = np.vstack(all_pts)
    labels = np.hstack(all_labels)

    # Normalise
    centroid = np.mean(pts, axis=0)
    pts -= centroid
    max_dist = np.max(np.linalg.norm(pts, axis=1))
    pts /= (max_dist + 1e-8)

    return pts.astype(np.float32), labels.astype(np.int64)

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np

class SyntheticPartDataset(Dataset):
    def __init__(self, data_list, num_points=2048, occlusion=True, crop_angle=120, dropout=0.3):
        self.data = data_list
        self.num_points = num_points
        self.occlusion = occlusion
        self.crop_angle = crop_angle
        self.dropout = dropout

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pts, labels = self.data[idx]
        pts = pts.copy()
        labels = labels.copy()

        if self.occlusion:
            pts, labels = self.simulate_occlusion(pts, labels)

        # Subsample to num_points
        if len(pts) > self.num_points:
            # Ensure positives (non-zero labels) are kept
            pos_mask = labels > 0
            pos_idx = np.where(pos_mask)[0]
            neg_idx = np.where(~pos_mask)[0]
            n_pos = len(pos_idx)
            n_neg_needed = max(0, self.num_points - n_pos)
            if n_neg_needed > 0 and len(neg_idx) > 0:
                chosen_neg = np.random.choice(neg_idx, n_neg_needed, replace=False)
                chosen = np.concatenate([pos_idx, chosen_neg])
            elif n_pos > 0:
                chosen = pos_idx
            else:
                chosen = np.random.choice(len(pts), self.num_points, replace=False)
            if len(chosen) < self.num_points:
                extra = np.random.choice(len(pts), self.num_points - len(chosen), replace=True)
                chosen = np.concatenate([chosen, extra])
        else:
            chosen = np.random.choice(len(pts), self.num_points, replace=True)

        pts = pts[chosen]
        labels = labels[chosen]

        return torch.tensor(pts, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

    def simulate_occlusion(self, points, labels):
        """Keep all non‑background points, occlude background from random viewpoint."""
        pos_mask = labels > 0
        pos_idx = np.where(pos_mask)[0]
        neg_idx = np.where(~pos_mask)[0]

        if len(neg_idx) > 0:
            neg_points = points[neg_idx]
            centroid = np.mean(neg_points, axis=0)
            centered = neg_points - centroid
            theta = np.random.uniform(0, 2*np.pi)
            phi = np.random.uniform(0, np.pi)
            view_dir = np.array([np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)])
            norms = np.linalg.norm(centered, axis=1)
            norms[norms == 0] = 1e-8
            dirs = centered / norms[:, np.newaxis]
            angles = np.arccos(np.clip(np.dot(dirs, view_dir), -1, 1))
            keep_mask = angles < np.radians(self.crop_angle/2)
            if self.dropout > 0:
                keep_neg = np.where(keep_mask)[0]
                n_keep = int(len(keep_neg) * (1 - self.dropout))
                if n_keep > 0 and len(keep_neg) > 0:
                    keep_neg = np.random.choice(keep_neg, n_keep, replace=False)
                    keep_mask = np.zeros(len(neg_points), dtype=bool)
                    keep_mask[keep_neg] = True
            neg_points = neg_points[keep_mask]
            neg_labels = labels[neg_idx][keep_mask]
        else:
            neg_points = np.empty((0, 3))
            neg_labels = np.empty(0, dtype=labels.dtype)

        pos_points = points[pos_idx]
        pos_labels = labels[pos_idx]

        if len(pos_points) > 0:
            new_points = np.concatenate([pos_points, neg_points], axis=0)
            new_labels = np.concatenate([pos_labels, neg_labels], axis=0)
        else:
            new_points = neg_points
            new_labels = neg_labels

        return new_points, new_labels

In [ ]:
import torch.nn as nn

def knn(x, k=20):
    B, _, N = x.shape
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]
    return idx

def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None:
        idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx = idx + idx_base
    idx = idx.view(-1)
    _, num_dims, _ = x.size()
    x = x.transpose(2, 1).contiguous()
    feature = x.view(B * N, -1)[idx, :]
    feature = feature.view(B, N, k, C)
    x = x.view(B, N, 1, C).repeat(1, 1, k, 1)
    feature = torch.cat((feature - x, x), dim=3)
    feature = feature.permute(0, 3, 1, 2).contiguous()
    return feature

class DGCNNPartSeg(nn.Module):
    def __init__(self, num_classes=7, k=20):
        super().__init__()
        self.k = k
        self.conv1 = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64*2, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64*2, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Conv1d(64 + 64 + 128 + 256, 256, 1, bias=False),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, num_classes, 1)
        )

    def forward(self, x):
        B, C, N = x.shape
        edge_feat = get_graph_feature(x, k=self.k)
        x1 = self.conv1(edge_feat).max(dim=-1, keepdim=False)[0]
        edge_feat = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge_feat).max(dim=-1, keepdim=False)[0]
        edge_feat = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge_feat).max(dim=-1, keepdim=False)[0]
        x_cat = torch.cat([x1, x2, x3], dim=1)
        x_global = x_cat.max(dim=-1, keepdim=True)[0].repeat(1, 1, N)
        x_concat = torch.cat([x1, x2, x3, x_global], dim=1)
        out = self.fc(x_concat).permute(0, 2, 1)
        return out

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pickle
from tqdm import tqdm


# Load synthetic data
with open("synthetic_parts.pkl", "rb") as f:
    data = pickle.load(f)

# Split train/val (80/20)
np.random.seed(42)
np.random.shuffle(data)
split = int(0.8 * len(data))
train_data = data[:split]
val_data = data[split:]

# Create datasets with occlusion
train_dataset = SyntheticPartDataset(train_data, num_points=2048, occlusion=True)
val_dataset   = SyntheticPartDataset(val_data, num_points=2048, occlusion=False)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DGCNNPartSeg(num_classes=7, k=20).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

NUM_EPOCHS = 30
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for pts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        pts = pts.permute(0, 2, 1).to(device)   # (B, 3, N)
        labels = labels.to(device)               # (B, N)
        optimizer.zero_grad()
        logits = model(pts)                      # (B, N, 7)
        loss = criterion(logits.permute(0, 2, 1), labels)  # CrossEntropy expects (B, C, N)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for pts, labels in val_loader:
            pts = pts.permute(0, 2, 1).to(device)
            labels = labels.to(device)
            logits = model(pts)
            preds = logits.argmax(dim=2)   # (B, N)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    val_acc = correct / total
    print(f"Epoch {epoch+1}: Train Loss {avg_loss:.4f}, Val Acc {val_acc:.4f}")

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_part_detector.pth")
        print(f"  -> Saved best model (acc {val_acc:.4f})")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

Epoch 1: 100%|██████████| 500/500 [01:08<00:00,  7.28it/s]


Epoch 1: Train Loss 1.0500, Val Acc 0.7816
  -> Saved best model (acc 0.7816)


Epoch 2: 100%|██████████| 500/500 [01:08<00:00,  7.32it/s]


Epoch 2: Train Loss 0.7810, Val Acc 0.7978
  -> Saved best model (acc 0.7978)


Epoch 3: 100%|██████████| 500/500 [01:08<00:00,  7.33it/s]


...

Epoch 9: Train Loss 0.3240, Val Acc 0.8331


Epoch 10: 100%|██████████| 500/500 [01:08<00:00,  7.32it/s]


Epoch 10: Train Loss 0.2936, Val Acc 0.8063


Epoch 11:  58%|█████▊    | 291/500 [00:39<00:28,  7.29it/s]


KeyboardInterrupt: 

In [ ]:
# Constants for real data (same as before)
AFFORDANCE_NAMES = ['grasp', 'contain', 'lift', 'openable', 'layable', 'sittable', 'support',
                    'wrap_grasp', 'pourable', 'move', 'displaY', 'pushable', 'pull', 'listen',
                    'wear', 'press', 'cut', 'stab']
SELECTED_CLASSES = ['move', 'support', 'sittable', 'contain', 'displaY']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_AFFORDANCES = len(SELECTED_CLASSES)

# Part‑to‑affordance mapping (same as before)
part_to_affordance = {
    1: [0],      # handle -> move
    2: [0],      # blade -> move
    3: [1, 2],   # flat surface -> support, sittable
    4: [3],      # concave -> contain
    5: [3],      # rim -> contain
    6: [4],      # displaY panel -> displaY
}

# Load real test data
import pickle
with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
all_categories = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = ['Scissors', 'Hammer', 'Laptop', 'Camera']
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_categories]
test_split = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES]
print(f"Test samples (zero-shot): {len(test_split)}")

# Helper to extract real affordance ground truth from a sample
def get_affordance_labels(sample):
    labels = []
    for aff_idx in SELECTED_INDICES:
        aff = AFFORDANCE_NAMES[aff_idx]
        mask = sample['full_shape']['label'][aff].flatten()
        mask = (mask > 0.5).astype(np.float32)
        labels.append(mask)
    return np.stack(labels, axis=1)  # (N, C)

# Load best part detector
model = DGCNNPartSeg(num_classes=7, k=20).to(device)
model.load_state_dict(torch.load("best_part_detector.pth", map_location=device))
model.eval()

iou_per_class = np.zeros(NUM_AFFORDANCES)
count_per_class = np.zeros(NUM_AFFORDANCES)

for sample in tqdm(test_split, desc="Evaluating zero-shot"):
    pts_full = sample['full_shape']['coordinate'].astype(np.float32)
    # Normalise as in training (optional, but helps)
    centroid = np.mean(pts_full, axis=0)
    pts_full -= centroid
    max_dist = np.max(np.linalg.norm(pts_full, axis=1))
    pts_full /= (max_dist + 1e-8)

    # Get real affordance labels (multi-hot)
    real_aff = get_affordance_labels(sample)

    # Run part detector on full point cloud (no occlusion, but you can also add occlusion later)
    pts_tensor = torch.tensor(pts_full).unsqueeze(0).permute(0, 2, 1).to(device)  # (1,3,N)
    with torch.no_grad():
        logits = model(pts_tensor)
        part_pred = logits.argmax(dim=2).squeeze(0).cpu().numpy()  # (N,)

    # Convert part predictions to affordance predictions (multi-hot)
    pred_aff = np.zeros((len(part_pred), NUM_AFFORDANCES), dtype=np.float32)
    for part_id, aff_ids in part_to_affordance.items():
        mask = (part_pred == part_id)
        for aff_id in aff_ids:
            pred_aff[mask, aff_id] = 1.0

    # Compute IoU per affordance class
    for c in range(NUM_AFFORDANCES):
        if real_aff[:, c].sum() > 0:
            inter = (pred_aff[:, c] * real_aff[:, c]).sum()
            union = (pred_aff[:, c] + real_aff[:, c]).clip(0, 1).sum()
            iou = inter / (union + 1e-8)
            iou_per_class[c] += iou
            count_per_class[c] += 1

avg_iou = iou_per_class / (count_per_class + 1e-8)
mean_iou = avg_iou.mean()

print(f"\nZero‑Shot Affordance Segmentation (via Geometric Parts)")
print(f"mIoU: {mean_iou:.4f}")
for i, name in enumerate(SELECTED_CLASSES):
    print(f"  {name}: {avg_iou[i]:.4f}")

Test samples (zero-shot): 344


Evaluating zero-shot: 100%|██████████| 344/344 [00:04<00:00, 85.27it/s]


Zero‑Shot Affordance Segmentation (via Geometric Parts)
mIoU: 0.0001
  move: 0.0000
  support: 0.0000
  sittable: 0.0000
  contain: 0.0000
  displaY: 0.0005


In [ ]:
import numpy as np
import open3d as o3d

def create_coherent_object():
    """Generate a synthetic object with meaningful parts attached."""
    obj_type = np.random.choice(['mug', 'bowl', 'knife', 'laptop', 'chair'])
    parts = []
    labels = []

    if obj_type == 'mug':
        # Body: cylinder
        r_body = np.random.uniform(0.15, 0.3)
        h_body = np.random.uniform(0.3, 0.6)
        body = o3d.geometry.TriangleMesh.create_cylinder(radius=r_body, height=h_body)
        body.translate([0, 0, 0])
        # Handle: thin torus or curved cylinder attached to side
        # Use a torus arc (simplified as a thin cylinder rotated and translated)
        r_handle = np.random.uniform(0.1, 0.15)
        t_handle = np.random.uniform(0.03, 0.05)
        handle = o3d.geometry.TriangleMesh.create_torus(torus_radius=r_handle, tube_radius=t_handle)
        # Rotate and move to side of body
        R = o3d.geometry.get_rotation_matrix_from_xyz((np.pi/2, 0, 0))
        handle.rotate(R)
        handle.translate([r_body + 0.05, 0, 0])
        # Combine
        mesh = body + handle
        # Sample points
        pcd = mesh.sample_points_uniformly(number_of_points=2048)
        pts = np.asarray(pcd.points)
        # Label points: simple heuristic based on distance to handle axis (for demonstration)
        # More accurate: compute per-vertex labels on mesh before sampling, but for speed we approximate.
        # We'll label points near the handle region as 1 (handle), rest body (0)
        labels = np.zeros(len(pts), dtype=int)
        # Heuristic: points with x > r_body are handle
        labels[pts[:, 0] > r_body*0.8] = 1
        # Also points in a ring around the torus
        # (This is imprecise; better to label before sampling, but okay for proof)
        # We'll refine later.
        # For now, just return all as body? No, we need proper labels.
        # We can label by checking distances to the handle mesh.
        # Instead, we'll label the handle mesh points separately then combine.
        return None  # placeholder, you'll need proper labelling

    # ... similar for other types

    # You need to generate points separately for each part to get correct labels.
    # Better approach: sample points from each part mesh, assign label, then combine.

In [ ]:
import torch
import numpy as np

def compute_geometric_labels_gpu(points_batch, k=30):
    """
    points_batch: tensor of shape (B, N, 3) – already normalized point clouds, on GPU.
    Returns labels: (B, N) tensor on GPU, with integer labels 0..6.
    """
    B, N, _ = points_batch.shape
    device = points_batch.device

    # 1. Distance matrix: (B, N, N)
    dists = torch.cdist(points_batch, points_batch)  # B,N,N

    # 2. For each point, find k nearest neighbors (excluding itself)
    #    We take k+1 smallest distances and ignore the first (which is self-distance = 0)
    topk_vals, topk_idx = dists.topk(k=k+1, dim=-1, largest=False)   # B, N, k+1
    # Remove the closest (self)
    neighbor_idx = topk_idx[:, :, 1:]   # B, N, k
    # Gather neighbor coordinates
    neighbors = torch.gather(points_batch.unsqueeze(2).expand(-1, -1, k, -1),
                             dim=1,
                             index=neighbor_idx.unsqueeze(-1).expand(-1, -1, -1, 3))
    # neighbors shape: (B, N, k, 3)

    # 3. Center each neighborhood
    centered = neighbors - neighbors.mean(dim=2, keepdim=True)   # B, N, k, 3

    # 4. Covariance matrices: (B, N, 3, 3)
    #    For each point, cov = (1/k) * centered.T @ centered
    #    centered: B, N, k, 3 -> transpose last two dims: B, N, 3, k, then matmul with B, N, k, 3
    cov = torch.matmul(centered.transpose(2, 3), centered) / k   # B, N, 3, 3

    # 5. Eigen decomposition (batched)
    eigvals, eigvecs = torch.linalg.eigh(cov)   # eigvals: B, N, 3 (ascending), eigvecs: B, N, 3, 3
    # Sort descending: reverse last dim
    eigvals = eigvals.flip(-1)  # now B, N, 3 descending
    # eigvecs correspondingly: flip the last dim (columns)
    eigvecs = eigvecs.flip(-1)

    lambda1 = eigvals[:, :, 0]
    lambda2 = eigvals[:, :, 1]
    lambda3 = eigvals[:, :, 2]
    sum_l = lambda1 + lambda2 + lambda3 + 1e-10

    linearity  = (lambda1 - lambda2) / sum_l
    planarity  = (lambda2 - lambda3) / sum_l
    scattering = lambda3 / sum_l

    # Curvature approximation: average distance to tangent plane (normal = smallest eigenvector before flip)
    # Since we flipped, the normal corresponds to the last eigenvector (original smallest)
    normal = eigvecs[:, :, :, 2]   # B, N, 3 (the direction of the smallest eigenvalue after sorting descending is index 2)
    # dot product with centered neighbors
    dists_to_plane = torch.abs(torch.sum(centered * normal.unsqueeze(2), dim=-1))  # B, N, k
    curvature = dists_to_plane.mean(dim=-1)  # B, N

    # 6. Label assignment (vectorized)
    labels = torch.zeros((B, N), dtype=torch.long, device=device)  # 0: body

    # Display panel (vertical flat): planarity > 0.6, curvature < 0.02, normal horizontal, lambda1 > 0.1
    mask_display = (planarity > 0.6) & (curvature < 0.02) & (torch.abs(normal[..., 2]) < 0.3) & (lambda1 > 0.1)
    labels[mask_display] = 6

    # Flat surface (horizontal): planarity > 0.6, curvature < 0.02, not display
    mask_flat = (planarity > 0.6) & (curvature < 0.02) & ~mask_display
    labels[mask_flat] = 3

    # Handle: linearity > 0.6 and lambda1 > 0.05
    mask_handle = (linearity > 0.6) & (lambda1 > 0.05) & (labels == 0)  # ensure not already labeled
    labels[mask_handle] = 1

    # Concave: curvature > 0.03 and planarity < 0.3
    mask_concave = (curvature > 0.03) & (planarity < 0.3) & (labels == 0)
    labels[mask_concave] = 4

    # Blade: planarity > 0.4 and curvature > 0.02
    mask_blade = (planarity > 0.4) & (curvature > 0.02) & (labels == 0)
    labels[mask_blade] = 2

    return labels

In [ ]:
import pickle
import numpy as np
import torch
from tqdm import tqdm

# -------------------------------------------------------------------
# 1. GPU‑accelerated geometric labelling (PyTorch)
# -------------------------------------------------------------------
def compute_geometric_labels_gpu(points_batch, k=30):
    """
    points_batch: tensor of shape (B, N, 3) on GPU.
    Returns labels: (B, N) tensor on GPU, integer labels 0..6.
    """
    B, N, _ = points_batch.shape
    device = points_batch.device

    # If N is too small, reduce k
    k = min(k, N-1)  # avoid topk error
    if k < 1:
        # cloud has 1 point – return all zeros
        return torch.zeros((B, N), dtype=torch.long, device=device)

    # Distance matrix: (B, N, N)
    dists = torch.cdist(points_batch, points_batch)

    # k nearest neighbours (excluding self)
    topk_vals, topk_idx = dists.topk(k=k+1, dim=-1, largest=False)   # B, N, k+1
    neighbor_idx = topk_idx[:, :, 1:]                                 # B, N, k
    # Gather neighbour coordinates: (B, N, k, 3)
    neighbors = torch.gather(points_batch.unsqueeze(2).expand(-1, -1, k, -1),
                             dim=1,
                             index=neighbor_idx.unsqueeze(-1).expand(-1, -1, -1, 3))

    # Center neighbourhoods
    centered = neighbors - neighbors.mean(dim=2, keepdim=True)        # B, N, k, 3

    # Covariance matrices: (B, N, 3, 3)
    cov = torch.matmul(centered.transpose(2, 3), centered) / k

    # Eigen decomposition (batched)
    eigvals, eigvecs = torch.linalg.eigh(cov)        # ascending order
    # Sort descending
    eigvals = eigvals.flip(-1)                       # B, N, 3
    eigvecs = eigvecs.flip(-1)                       # corresponding eigenvectors

    lambda1 = eigvals[:, :, 0]
    lambda2 = eigvals[:, :, 1]
    lambda3 = eigvals[:, :, 2]
    sum_l = lambda1 + lambda2 + lambda3 + 1e-10

    linearity  = (lambda1 - lambda2) / sum_l
    planarity  = (lambda2 - lambda3) / sum_l
    scattering = lambda3 / sum_l

    # Normal = smallest eigenvector (original last after flipping -> index 2)
    normal = eigvecs[:, :, :, 2]    # B, N, 3
    dists_to_plane = torch.abs(torch.sum(centered * normal.unsqueeze(2), dim=-1))  # B, N, k
    curvature = dists_to_plane.mean(dim=-1)   # B, N

    # Rule‑based label assignment (vectorized)
    labels = torch.zeros((B, N), dtype=torch.long, device=device)

    # Display panel (vertical flat)
    mask_disp = (planarity > 0.6) & (curvature < 0.02) & (torch.abs(normal[..., 2]) < 0.3) & (lambda1 > 0.1)
    labels[mask_disp] = 6

    # Horizontal flat surface
    mask_flat = (planarity > 0.6) & (curvature < 0.02) & ~mask_disp
    labels[mask_flat] = 3

    # Handle (elongated)
    mask_handle = (linearity > 0.6) & (lambda1 > 0.05) & (labels == 0)
    labels[mask_handle] = 1

    # Concave
    mask_concave = (curvature > 0.03) & (planarity < 0.3) & (labels == 0)
    labels[mask_concave] = 4

    # Blade
    mask_blade = (planarity > 0.4) & (curvature > 0.02) & (labels == 0)
    labels[mask_blade] = 2

    return labels


# -------------------------------------------------------------------
# 2. Load and prepare data (same as before)
# -------------------------------------------------------------------
with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)

TEST_CATEGORIES = ['Scissors', 'Hammer', 'Laptop', 'Camera']
all_categories = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_categories]
train_data_real = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES]
print(f"Real training samples (excluding test categories): {len(train_data_real)}")

# -------------------------------------------------------------------
# 3. Normalise all point clouds and collect them
# -------------------------------------------------------------------
norm_points = []   # list of numpy arrays, each (N_i, 3)
for sample in tqdm(train_data_real, desc="Normalising"):
    pts = sample['full_shape']['coordinate'].astype(np.float32)
    centroid = np.mean(pts, axis=0)
    pts -= centroid
    max_dist = np.max(np.linalg.norm(pts, axis=1))
    pts /= (max_dist + 1e-8)
    norm_points.append(pts)

# -------------------------------------------------------------------
# 4. GPU processing – one cloud at a time (no padding, no contamination)
# -------------------------------------------------------------------
device = torch.device('cuda')
geom_dataset = []

for pts in tqdm(norm_points, desc="GPU labelling"):
    # Move to GPU with batch dimension 1
    tensor = torch.from_numpy(pts).unsqueeze(0).to(device)  # (1, N, 3)
    with torch.no_grad():
        lbl_batch = compute_geometric_labels_gpu(tensor, k=30)  # (1, N)
    labels = lbl_batch.squeeze(0).cpu().numpy()               # (N,)
    geom_dataset.append((pts, labels))

# -------------------------------------------------------------------
# 5. Save the result
# -------------------------------------------------------------------
with open("geom_real_dataset.pkl", "wb") as f:
    pickle.dump(geom_dataset, f)
print(f"Saved {len(geom_dataset)} geometrically labeled real point clouds.")

Real training samples (excluding test categories): 15738


GPU labelling: 100%|██████████| 15738/15738 [00:45<00:00, 348.80it/s]


Saved 15738 geometrically labeled real point clouds.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm

# ------------------------------------------------------------
# Dataset for real geometry part labels
# ------------------------------------------------------------
class RealPartDataset(Dataset):
    def __init__(self, data_list, num_points=2048, occlusion=True, crop_angle=120, dropout=0.3):
        self.data = data_list          # list of (pts, labels) numpy arrays
        self.num_points = num_points
        self.occlusion = occlusion
        self.crop_angle = crop_angle
        self.dropout = dropout

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pts, labels = self.data[idx]
        pts = pts.copy()
        labels = labels.copy()

        if self.occlusion:
            pts, labels = self.simulate_occlusion(pts, labels)

        # Subsample to exactly num_points while keeping positives
        if len(pts) > self.num_points:
            pos_mask = labels > 0
            pos_idx = np.where(pos_mask)[0]
            neg_idx = np.where(~pos_mask)[0]
            n_pos = len(pos_idx)
            n_neg_needed = max(0, self.num_points - n_pos)

            if n_neg_needed > 0 and len(neg_idx) > 0:
                chosen_neg = np.random.choice(neg_idx, n_neg_needed, replace=False)
                chosen = np.concatenate([pos_idx, chosen_neg])
            elif n_pos > 0:
                chosen = pos_idx
            else:
                chosen = np.random.choice(len(pts), self.num_points, replace=False)

            if len(chosen) < self.num_points:
                extra = np.random.choice(len(pts), self.num_points - len(chosen), replace=True)
                chosen = np.concatenate([chosen, extra])
        else:
            chosen = np.random.choice(len(pts), self.num_points, replace=True)

        pts = pts[chosen]
        labels = labels[chosen]

        return torch.tensor(pts, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

    def simulate_occlusion(self, points, labels):
        """Keep all non‑background points, occlude background from random viewpoint."""
        pos_mask = labels > 0
        pos_idx = np.where(pos_mask)[0]
        neg_idx = np.where(~pos_mask)[0]

        if len(neg_idx) > 0:
            neg_points = points[neg_idx]
            centroid = np.mean(neg_points, axis=0)
            centered = neg_points - centroid
            theta = np.random.uniform(0, 2 * np.pi)
            phi = np.random.uniform(0, np.pi)
            view_dir = np.array([np.sin(phi) * np.cos(theta),
                                 np.sin(phi) * np.sin(theta),
                                 np.cos(phi)])
            norms = np.linalg.norm(centered, axis=1)
            norms[norms == 0] = 1e-8
            dirs = centered / norms[:, np.newaxis]
            angles = np.arccos(np.clip(np.dot(dirs, view_dir), -1, 1))
            keep_mask = angles < np.radians(self.crop_angle / 2)

            if self.dropout > 0:
                keep_neg = np.where(keep_mask)[0]
                n_keep = int(len(keep_neg) * (1 - self.dropout))
                if n_keep > 0 and len(keep_neg) > 0:
                    keep_neg = np.random.choice(keep_neg, n_keep, replace=False)
                    keep_mask = np.zeros(len(neg_points), dtype=bool)
                    keep_mask[keep_neg] = True

            neg_points = neg_points[keep_mask]
            neg_labels = labels[neg_idx][keep_mask]
        else:
            neg_points = np.empty((0, 3))
            neg_labels = np.empty(0, dtype=labels.dtype)

        pos_points = points[pos_idx]
        pos_labels = labels[pos_idx]

        if len(pos_points) > 0:
            new_points = np.concatenate([pos_points, neg_points], axis=0)
            new_labels = np.concatenate([pos_labels, neg_labels], axis=0)
        else:
            new_points = neg_points
            new_labels = neg_labels

        return new_points, new_labels


# ------------------------------------------------------------
# Load real geometry dataset and create DataLoaders
# ------------------------------------------------------------
with open("geom_real_dataset.pkl", "rb") as f:
    real_geom_data = pickle.load(f)

np.random.seed(42)
np.random.shuffle(real_geom_data)
split = int(0.8 * len(real_geom_data))
train_data = real_geom_data[:split]
val_data   = real_geom_data[split:]

batch_size = 8
train_dataset = RealPartDataset(train_data, num_points=2048, occlusion=True)
val_dataset   = RealPartDataset(val_data,   num_points=2048, occlusion=False)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2)

# ------------------------------------------------------------
# Model, loss, optimizer
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DGCNNPartSeg(num_classes=7, k=20).to(device)   # must be already defined
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

NUM_EPOCHS = 20
best_val_acc = 0.0   # <-- the missing line

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for pts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        pts = pts.permute(0, 2, 1).to(device)   # (B, 3, N)
        labels = labels.to(device)               # (B, N)
        optimizer.zero_grad()
        logits = model(pts)                      # (B, N, 7)
        loss = criterion(logits.permute(0, 2, 1), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for pts, labels in val_loader:
            pts = pts.permute(0, 2, 1).to(device)
            labels = labels.to(device)
            logits = model(pts)
            preds = logits.argmax(dim=2)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    val_acc = correct / total
    print(f"Epoch {epoch+1}: Train Loss {avg_loss:.4f}, Val Acc {val_acc:.4f}")

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_part_detector.pth")
        print(f"  -> Saved best model (acc {val_acc:.4f})")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

Epoch 1:   3%|▎         | 45/1574 [00:06<03:42,  6.88it/s]


ValueError: Caught ValueError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_1934/677022709.py", line 50, in __getitem__
    chosen = np.random.choice(len(pts), self.num_points, replace=True)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "numpy/random/mtrand.pyx", line 968, in numpy.random.mtrand.RandomState.choice
ValueError: a must be greater than 0 unless no samples are taken


In [ ]:
import pickle
import numpy as np
import torch
from tqdm import tqdm

# ------------------------------
# Constants (same as before)
# ------------------------------
AFFORDANCE_NAMES = ['grasp', 'contain', 'lift', 'openable', 'layable', 'sittable', 'support',
                    'wrap_grasp', 'pourable', 'move', 'displaY', 'pushable', 'pull', 'listen',
                    'wear', 'press', 'cut', 'stab']
SELECTED_CLASSES = ['move', 'support', 'sittable', 'contain', 'displaY']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_AFFORDANCES = len(SELECTED_CLASSES)

part_to_affordance = {
    1: [0],      # handle -> move
    2: [0],      # blade -> move
    3: [1, 2],   # flat surface -> support, sittable
    4: [3],      # concave -> contain
    5: [3],      # rim -> contain
    6: [4],      # displaY panel -> displaY
}

# ------------------------------
# Load test split
# ------------------------------
with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
all_categories = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = ['Scissors', 'Hammer', 'Laptop', 'Camera']
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_categories]
test_split = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES]
print(f"Test samples (zero-shot): {len(test_split)}")

def get_affordance_labels(sample):
    labels = []
    for aff_idx in SELECTED_INDICES:
        aff = AFFORDANCE_NAMES[aff_idx]
        mask = sample['full_shape']['label'][aff].flatten()
        mask = (mask > 0.5).astype(np.float32)
        labels.append(mask)
    return np.stack(labels, axis=1)

# ------------------------------
# Load best real‑geometry part detector
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DGCNNPartSeg(num_classes=7, k=20).to(device)
model.load_state_dict(torch.load("best_part_detector.pth", map_location=device))
model.eval()

iou_per_class = np.zeros(NUM_AFFORDANCES)
count_per_class = np.zeros(NUM_AFFORDANCES)

for sample in tqdm(test_split, desc="Evaluating zero-shot"):
    pts_full = sample['full_shape']['coordinate'].astype(np.float32)
    centroid = np.mean(pts_full, axis=0)
    pts_full -= centroid
    max_dist = np.max(np.linalg.norm(pts_full, axis=1))
    pts_full /= (max_dist + 1e-8)

    real_aff = get_affordance_labels(sample)

    pts_tensor = torch.tensor(pts_full).unsqueeze(0).permute(0, 2, 1).to(device)
    with torch.no_grad():
        logits = model(pts_tensor)
        part_pred = logits.argmax(dim=2).squeeze(0).cpu().numpy()

    pred_aff = np.zeros((len(part_pred), NUM_AFFORDANCES), dtype=np.float32)
    for part_id, aff_ids in part_to_affordance.items():
        mask = (part_pred == part_id)
        for aff_id in aff_ids:
            pred_aff[mask, aff_id] = 1.0

    for c in range(NUM_AFFORDANCES):
        if real_aff[:, c].sum() > 0:
            inter = (pred_aff[:, c] * real_aff[:, c]).sum()
            union = (pred_aff[:, c] + real_aff[:, c]).clip(0, 1).sum()
            iou = inter / (union + 1e-8)
            iou_per_class[c] += iou
            count_per_class[c] += 1

avg_iou = iou_per_class / (count_per_class + 1e-8)
mean_iou = avg_iou.mean()

print(f"\nZero‑Shot Affordance Segmentation (via Real‑Geometry Part Detector)")
print(f"mIoU: {mean_iou:.4f}")
for i, name in enumerate(SELECTED_CLASSES):
    print(f"  {name}: {avg_iou[i]:.4f}")

Epoch 1:   2%|▏         | 36/1574 [00:06<04:48,  5.34it/s]


KeyboardInterrupt: 